In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from imblearn.over_sampling import SMOTE


c:\Users\vivek\anaconda4\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")
df = df.drop("customerID",axis=1)
# Inspect problematic TotalCharges values
whitespace_count = (df['TotalCharges'] == " ").sum()
print(f"TotalCharges whitespace entries before conversion: {whitespace_count}")

# Convert TotalCharges to numeric, whitespace -> NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verify
print("NaN in TotalCharges after conversion:", df['TotalCharges'].isna().sum())

# Convert SeniorCitizen to categorical
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})
df.head()

TotalCharges whitespace entries before conversion: 11
NaN in TotalCharges after conversion: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   object 
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [4]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5634, 19)
Test shape: (1409, 19)


In [5]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]

binary_cols = [col for col in categorical_cols if X_train[col].nunique() == 2]
multi_cols  = [col for col in categorical_cols if X_train[col].nunique() > 2]

print("Numeric:", numeric_cols)
print("Binary categorical:", binary_cols)
print("Multi categorical:", multi_cols)


Numeric: ['tenure', 'MonthlyCharges', 'TotalCharges']
Binary categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi categorical: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']


In [6]:
print("\nNaN counts before imputation:")
print(X_train[numeric_cols + categorical_cols].isna().sum())

# Numeric imputer
num_imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = num_imputer.transform(X_test[numeric_cols])

# Categorical imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols]  = cat_imputer.transform(X_test[categorical_cols])

print("\nNaN counts after imputation:")
print(X_train[numeric_cols + categorical_cols].isna().sum())



NaN counts before imputation:
tenure              0
MonthlyCharges      0
TotalCharges        8
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
dtype: int64

NaN counts after imputation:
tenure              0
MonthlyCharges      0
TotalCharges        0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
dtype: int64


In [7]:
print("\n=== BEFORE OUTLIER CLIPPING ===")
for col in numeric_cols:
    print(f"{col}: min={X_train[col].min()}, max={X_train[col].max()}")

# Clip using IQR from training data
X_train_clipped = X_train.copy()
X_test_clipped  = X_test.copy()

for col in numeric_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    X_train_clipped[col] = X_train[col].clip(lower, upper)
    X_test_clipped[col]  = X_test[col].clip(lower, upper)

print("\n=== AFTER OUTLIER CLIPPING ===")
for col in numeric_cols:
    print(f"{col}: min={X_train_clipped[col].min()}, max={X_train_clipped[col].max()}")



=== BEFORE OUTLIER CLIPPING ===
tenure: min=0.0, max=72.0
MonthlyCharges: min=18.4, max=118.75
TotalCharges: min=18.85, max=8684.8

=== AFTER OUTLIER CLIPPING ===
tenure: min=0.0, max=72.0
MonthlyCharges: min=18.4, max=118.75
TotalCharges: min=18.85, max=8684.8


In [8]:
# Label encode binary categorical columns
le = LabelEncoder()
for col in binary_cols:
    X_train_clipped[col] = le.fit_transform(X_train_clipped[col])
    X_test_clipped[col]  = le.transform(X_test_clipped[col])

# One-hot encode multi-class categorical columns
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train_clipped[multi_cols])
X_test_ohe  = ohe.transform(X_test_clipped[multi_cols])

X_train_enc = pd.concat([
    X_train_clipped.drop(multi_cols, axis=1),
    pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(multi_cols), index=X_train_clipped.index)
], axis=1)

X_test_enc = pd.concat([
    X_test_clipped.drop(multi_cols, axis=1),
    pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(multi_cols), index=X_test_clipped.index)
], axis=1)


In [9]:
scaler = StandardScaler()
X_train_enc[numeric_cols] = scaler.fit_transform(X_train_enc[numeric_cols])
X_test_enc[numeric_cols]  = scaler.transform(X_test_enc[numeric_cols])


In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=300),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": xgb.XGBClassifier(eval_metric='logloss')
}

cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = []
    for train_idx, val_idx in skf.split(X_train_enc, y_train):
        X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Apply SMOTE ONLY on training fold
        sm = SMOTE(random_state=42)
        X_tr_bal, y_tr_bal = sm.fit_resample(X_tr, y_tr)
        
        model.fit(X_tr_bal, y_tr_bal)
        y_pred = model.predict(X_val)
        score = recall_score(y_val, y_pred)
        scores.append(score)
    
    cv_results[name] = np.mean(scores)
    print(f"{name} CV Recall: {cv_results[name]:.4f}")


Logistic Regression CV Recall: 0.7766
Decision Tree CV Recall: 0.5478
Random Forest CV Recall: 0.5933
XGBoost CV Recall: 0.5880


In [11]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

best_score = 0
best_params = None
rf = RandomForestClassifier(random_state=42)

for n_estimators in param_grid_rf['n_estimators']:
    for max_depth in param_grid_rf['max_depth']:
        for min_samples_split in param_grid_rf['min_samples_split']:
            scores = []
            for train_idx, val_idx in skf.split(X_train_enc, y_train):
                X_tr, X_val = X_train_enc.iloc[train_idx], X_train_enc.iloc[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                
                sm = SMOTE(random_state=42)
                X_tr_bal, y_tr_bal = sm.fit_resample(X_tr, y_tr)
                
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    random_state=42
                )
                model.fit(X_tr_bal, y_tr_bal)
                y_pred_val = model.predict(X_val)
                scores.append(recall_score(y_val, y_pred_val))
            
            mean_score = np.mean(scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = {
                    'n_estimators': n_estimators,
                    'max_depth': max_depth,
                    'min_samples_split': min_samples_split
                }

print("Best RF params:", best_params)
print("Best CV Recall:", best_score)


Best RF params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2}
Best CV Recall: 0.7217391304347827


In [12]:
# Apply SMOTE once on full training set
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_enc, y_train)

final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train_bal, y_train_bal)

# Evaluate on untouched test set
y_pred = final_model.predict(X_test_enc)
print("\n=== FINAL TEST SET RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))



=== FINAL TEST SET RESULTS ===
Accuracy: 0.7636621717530163
Recall: 0.7219251336898396

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.78      0.83      1035
           1       0.54      0.72      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409

